In [ ]:
!pip install transformers datasets -q
!pip install transformers -q
!pip install keras_nlp -q
!pip install datasets -q
!pip install huggingface-hub -q
!pip install nltk -q
!pip install rouge-score -q
!pip install huggingface_hub
!pip install rouge-score -q
!pip install datasets -q
!pip install evaluate -q

import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer, DataCollatorForSeq2Seq
from transformers import AdamW
from datasets import Dataset
from transformers import get_scheduler
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer
from sklearn.metrics import average_precision_score
import gc
import string
from string import punctuation
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from torch.cuda.amp import autocast, GradScaler
from transformers import PegasusConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-nvrtc-cu12==1

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
class PegasusMemoryConfig(PegasusConfig):
    def __init__(self, n_gram_size=3, max_memory_size=10000, **kwargs):
        super().__init__(**kwargs)
        self.n_gram_size = n_gram_size
        self.max_memory_size = max_memory_size

class PegasusWithMemory(PegasusForConditionalGeneration):
    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, *model_args, **kwargs):
        # First load the base model configuration
        config = kwargs.pop('config', None)
        if config is None:
            config = PegasusMemoryConfig.from_pretrained(pretrained_model_name_or_path)

        # Ensure memory-specific parameters are set
        if not hasattr(config, 'n_gram_size'):
            config.n_gram_size = 3
        if not hasattr(config, 'max_memory_size'):
            config.max_memory_size = 10000

        # Initialize model with the configuration
        model = super().from_pretrained(
            pretrained_model_name_or_path,
            config=config,
            *model_args,
            **kwargs
        )

        # Initialize memory-specific attributes
        model.memory = {}
        model.tokenizer = PegasusTokenizer.from_pretrained(pretrained_model_name_or_path)

        return model

    def remove_punctuation(self, text):
        if isinstance(text, float):
            return text
        return "".join([char for char in text if char not in string.punctuation])

    def generate_n_grams(self, text, n):
        text = self.remove_punctuation(text)
        words = [word for word in text.split(" ") if word not in set(stopwords.words('english'))]
        n_grams = zip(*[words[i:] for i in range(n)])
        return [' '.join(ngram) for ngram in n_grams]

    def update_memory(self, text, input_data):
        ngrams = self.generate_n_grams(text, self.config.n_gram_size)
        filtered_input = " ".join([word for word in input_data.split(" ") if word not in set(stopwords.words('english'))])

        for ngram in ngrams:
            if ngram not in self.memory:
                self.memory[ngram] = set()
            self.memory[ngram].add(filtered_input)

            if len(self.memory[ngram]) > self.config.max_memory_size:
                self.memory[ngram].remove(next(iter(self.memory[ngram])))

    def retrieve_memory(self, text):
        ngrams = self.generate_n_grams(text, self.config.n_gram_size)
        relevant_data = []

        for ngram in ngrams:
            if ngram in self.memory:
                relevant_data.extend(self.memory[ngram])

        if relevant_data:
            return self.prepare_retrieval_data(list(set(relevant_data)), text)
        return []

    def prepare_retrieval_data(self, corpus, query, top_k=2):
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(corpus + [query])
        query_vector = tfidf_matrix[-1]

        similarities = cosine_similarity(query_vector, tfidf_matrix[:-1])
        top_k_indices = similarities.argsort()[0][-top_k:][::-1]
        return [corpus[idx] for idx in top_k_indices]

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        if input_ids is not None:
            text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)[0]
            memory_data = self.retrieve_memory(text)

            if memory_data:
                augmented_text = memory_data[0] + "|---|" + text
                inputs = self.tokenizer(
                    augmented_text,
                    truncation=True,
                    padding=True,
                    max_length=self.config.max_position_embeddings,
                    return_tensors="pt"
                ).to(input_ids.device)

                input_ids = inputs["input_ids"]
                attention_mask = inputs["attention_mask"]

            data = ""
            counter = 0
            for word in text.split(" "):
                if counter < 20:
                    if word not in set(stopwords.words('english')) and word != "":
                        data += word + " "
                        counter += 1
                else:
                    break

            self.update_memory(text, data)

        return super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs
        )

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        if input_ids is not None:
            text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)[0]
            memory_data = self.retrieve_memory(text)

            if memory_data:
                augmented_text = memory_data[0] + "|--|" + text
                inputs = self.tokenizer(
                    augmented_text,
                    truncation=True,
                    padding=True,
                    max_length=self.config.max_position_embeddings,
                    return_tensors="pt"
                ).to(input_ids.device)

                input_ids = inputs["input_ids"]
                attention_mask = inputs["attention_mask"]

            data = ""
            counter = 0
            for word in text.split(" "):
                if counter < 20:
                    if word not in set(stopwords.words('english')) and word != "":
                        data += word + " "
                        counter += 1
                else:
                    break
            self.update_memory(text, data)

        return super().generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )

# Modified Model
# model = PegasusWithMemory.from_pretrained("google/pegasus-cnn_dailymail")
# tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-cnn_dailymail')
model = PegasusWithMemory.from_pretrained("google/pegasus-xsum")
tokenizer = PegasusTokenizer.from_pretrained('google/pegasus-xsum')

# Original Model
# model_name = "google/pegasus-xsum"
# # model_name = "google/pegasus-cnn_dailymail"
# tokenizer = PegasusTokenizer.from_pretrained(model_name)
# model = PegasusForConditionalGeneration.from_pretrained(model_name)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusWithMemory were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

In [ ]:
from torch.optim import AdamW
from nltk.translate.bleu_score import sentence_bleu
from sklearn.metrics import average_precision_score
from datasets import load_dataset

import pandas as pd
from datasets import Dataset

# CNN-DailyNews
data = load_dataset("cnn_dailymail", "3.0.0")

train_df = data["train"]
val_df = data["validation"]
test_df = data["test"]

# sub_dataset = train_df.train_test_split(train_size=0.005, seed=42)
train_dataset = train_df.shuffle(seed=42).select(range(60))
val_dataset = val_df.shuffle(seed=42).select(range(20))
test_dataset = test_df.shuffle(seed=42).select(range(20))

print(train_dataset)
print(val_dataset)
print(test_dataset)

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 60
})
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 20
})
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 20
})


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import get_scheduler

import torch
from torch.utils.data import DataLoader
from transformers import get_scheduler

def train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, device, epochs=1, batch_size=1, grad_accum_steps=4):
    model.to(device)
    model.train()

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    num_training_steps = epochs * len(train_dataloader) // grad_accum_steps  # Adjust for grad accumulation
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=10,
        num_training_steps=num_training_steps
    )

    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        for step, batch in enumerate(train_dataloader):
            context = batch["article"]
            reference = batch["highlights"]

            inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt")
            labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids

            inputs, labels = inputs.to(device), labels.to(device)
            labels[labels == tokenizer.pad_token_id] = -100  # Ignore padding tokens

            optimizer.zero_grad(set_to_none=True)  # Free memory efficiently

            with torch.amp.autocast("cuda"):  # Fixed deprecation warning
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss / grad_accum_steps  # Normalize loss for accumulation

            scaler.scale(loss).backward()

            if (step + 1) % grad_accum_steps == 0:  # Only update every `grad_accum_steps`
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item() * grad_accum_steps  # Reverse normalization

            # Free memory
            del inputs, labels, outputs, loss
            torch.cuda.empty_cache()

        avg_train_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1} Training Loss = {avg_train_loss:.4f}")

        # Validate model after each epoch
        val_loss = evaluate_model(val_dataloader, model, tokenizer, device)
        print(f"Epoch {epoch+1} Validation Loss = {val_loss:.4f}")

    print("Training Completed")

def evaluate_model(val_dataloader, model, tokenizer, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in val_dataloader:
            context = batch["article"]
            reference = batch["highlights"]

            inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt")
            labels = tokenizer(reference, truncation=True, padding="max_length", return_tensors="pt").input_ids

            inputs, labels = inputs.to(device), labels.to(device)
            labels[labels == tokenizer.pad_token_id] = -100

            with torch.amp.autocast("cuda"):
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss

            total_loss += loss.item()

            del inputs, labels, outputs, loss
            torch.cuda.empty_cache()

    return total_loss / len(val_dataloader)

In [ ]:
import torch
from rouge_score import rouge_scorer

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.metrics import precision
from collections import Counter

def calculate_metrics_on_csv(test_dataset, model, tokenizer, device="cuda"):
    model.eval()
    model.to(device)

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    predictions = []
    references = []

    with torch.no_grad():
        for batch in test_dataset:
            # CNN-DailyMail News

            context = batch["article"]
            reference = batch["highlights"]

            # Custom CSV Data

            # category = batch["Category"]
            # query = batch["Headline"]
            # context = batch["Content"]
            # reference = batch["Human Summary"]

            # Tokenize inputs and generate summaries
            inputs = tokenizer(context, truncation=True, padding="max_length", return_tensors="pt").to(device)
            summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
            generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

            # Collect predictions and references
            predictions.append(generated_text)
            references.append(reference)

    rouge_scores = { "rouge1": [], "rouge2": [], "rougeL": [] }
    precision_scores = []

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge_scores["rouge1"].append(scores["rouge1"].fmeasure)
        rouge_scores["rouge2"].append(scores["rouge2"].fmeasure)
        rouge_scores["rougeL"].append(scores["rougeL"].fmeasure)

        # **Average Precision Calculation (Word-Level)**
        ref_words = Counter(ref.split())
        pred_words = Counter(pred.split())
        common_words = sum((ref_words & pred_words).values())  # Correct words retrieved
        total_predicted_words = sum(pred_words.values())  # Total words in generated summary
        precision_score = common_words / total_predicted_words if total_predicted_words > 0 else 0
        precision_scores.append(precision_score)

        ref_tokens = [ref.split()]  # BLEU expects a list of reference token lists
        pred_tokens = pred.split()

    avg_rouge_scores = {key: sum(values) / len(values) for key, values in rouge_scores.items()}
    avg_precision = sum(precision_scores) / len(precision_scores)

    print(f"Average ROUGE Scores: {avg_rouge_scores}")
    print(f"Average Precision: {avg_precision:.4f}")

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
train_loss = train_model_on_csv(train_dataset, val_dataset, model, tokenizer, optimizer, device, epochs=3, batch_size=1, grad_accum_steps=1)

Epoch 1 Training Loss = 3.2055
Epoch 1 Validation Loss = 2.3502
Epoch 2 Training Loss = 2.2872
Epoch 2 Validation Loss = 2.2547
Epoch 3 Training Loss = 2.2000
Epoch 3 Validation Loss = 2.2239
Training Completed


In [ ]:
calculate_metrics_on_csv(test_dataset, model, tokenizer)

Average ROUGE Scores: {'rouge1': 0.25158340590576606, 'rouge2': 0.09898588682638937, 'rougeL': 0.19701779803559424}
Average Precision: 0.3447


In [ ]:
text = """India successfully launched Chandrayaan-4, aiming to study the lunar surface and uncover its secrets."""

inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Output: {generated_text}")

Output: India's space agency has successfully launched its fourth mission to the Moon.


In [ ]:
text = """Jasvir Singh Garhi, Punjab BSP president,  claims the BSP-SAD alliance will prioritize justice in sacrilege cases, combat corruption, and improve infrastructure if elected.  Key election issues include employment,  drug control, and farmer welfare.  Garhi dismisses concerns about internal party dissent and ticket distribution,  predicting a strong showing, especially in the Doaba region, and refuting  opinion polls favoring AAP or Congress. He criticizes the Congress government's performance and alleges the party is using Dalit leader Channi for political gain."""

inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Output: {generated_text}")

Output: The Bahujan Samaj Party (BSP) and Shiromani Akali Dal (SAD) have formed an alliance to fight the upcoming Punjab elections.


In [ ]:
text = """India successfully launched Chandrayaan-4, aiming to study the lunar surface and uncover its secrets."""

inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Output: {generated_text}")

Output: India successfully launched Chandrayaan-4, aiming to study the lunar surface and uncover secrets.


In [ ]:
text = """Jasvir Singh Garhi, Punjab BSP president,  claims the BSP-SAD alliance will prioritize justice in sacrilege cases, combat corruption, and improve infrastructure if elected.  Key election issues include employment,  drug control, and farmer welfare.  Garhi dismisses concerns about internal party dissent and ticket distribution,  predicting a strong showing, especially in the Doaba region, and refuting  opinion polls favoring AAP or Congress. He criticizes the Congress government's performance and alleges the party is using Dalit leader Channi for political gain."""

inputs = tokenizer(text, truncation=True, padding="max_length", return_tensors="pt").to(device)
summary_ids = model.generate(**inputs, max_length=150, num_beams=4)
generated_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Output: {generated_text}")

Output: Jasvir Singh Garhi, Punjab BSP president, claims BSP-SAD alliance will prioritize justice sacrilege cases, combat corruption, improve infrastructure elected.
